# FastSpeech 2 - full pipeline on a GPU (Kaggle or Colab)

Runs everything for **one language**: download -> MFA alignment -> features -> training -> test-set inference,
and packs the checkpoint, logs, plots and audio into `outputs_<Language>.zip`.
`notebooks/english_gpu.ipynb` and `notebooks/telugu_gpu.ipynb` differ only in `LANGUAGE`; both can run at the same time.

**Kaggle** (recommended, keeps running with the browser closed)
1. New Notebook -> File -> Import Notebook -> Link -> the raw GitHub URL of this file.
2. Settings: Accelerator = GPU T4 (or P100), Internet = on (needs a phone-verified account).
3. English only: accept the terms at https://huggingface.co/datasets/ai4bharat/Svarah, then
   Add-ons -> Secrets -> add `HF_TOKEN` (a read token from https://huggingface.co/settings/tokens) and tick it for this notebook.
4. Save Version -> "Save & Run All (Commit)". When it finishes, `outputs_<Language>.zip` is in the Output tab.

**Colab**: Runtime -> Change runtime type -> T4 GPU. For English, add `HF_TOKEN` under Secrets (key icon). Runtime -> Run all,
and keep the tab open. The zip is downloaded at the end.

Unzip the result into the repository root; the paths inside match the repo layout.

In [ ]:
LANGUAGE = "Telugu"     # "English" or "Telugu"
TRAIN_STEPS = 60000     # upper bound; training also stops after TRAIN_HOURS
TRAIN_HOURS = 6.0       # Kaggle sessions are capped at 12 h, preprocessing + MFA take ~1-1.5 h
REPO = "https://github.com/shubham-pattewar/FastSpeech.git"

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

assert LANGUAGE in ("English", "Telugu")
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = "/tmp/FastSpeech"  # code + data live outside the saved output folder
NPROC = os.cpu_count()

if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
os.chdir(CODE)
LOGS = f"{LANGUAGE}/logs"
os.makedirs(LOGS, exist_ok=True)
os.environ["TQDM_MININTERVAL"] = "30"


def run(cmd, log=None):
    """Run a shell command, stream its output and append it to a log file."""
    print("$", cmd, flush=True)
    f = open(os.path.join(LOGS, log), "a") if log else None
    with subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            if f:
                f.write(line)
    if f:
        f.close()
    if p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)
print("cpus:", NPROC)

In [ ]:
run("pip install -q librosa soundfile tgt g2p_en inflect pyworld pyarrow")

token = None
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    elif ON_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", e)
if token:
    os.environ["HF_TOKEN"] = token
elif LANGUAGE == "English":
    raise RuntimeError("Svarah is gated: accept its terms on HuggingFace and add an HF_TOKEN secret")

Montreal Forced Aligner in its own micromamba environment (a few minutes).

In [ ]:
run("cd /tmp && [ -x /tmp/bin/micromamba ] || curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba")
run("/tmp/bin/micromamba create -y -q -r /tmp/mamba -n mfa -c conda-forge montreal-forced-aligner python=3.11 > /dev/null")
MFA = "/tmp/bin/micromamba run -r /tmp/mamba -n mfa mfa"
run(f"{MFA} version")

In [ ]:
from huggingface_hub import snapshot_download

repo_id, local = {
    "English": ("ai4bharat/Svarah", "data/English/Svarah_hf"),
    "Telugu": ("SPRINGLab/IndicTTS_Telugu", "data/Telugu/IndicTTS_Telugu"),
}[LANGUAGE]
snapshot_download(repo_id, repo_type="dataset", local_dir=local, allow_patterns=["*.parquet", "README.md"])

os.makedirs("hifigan", exist_ok=True)
if not os.path.exists("hifigan/generator_universal.pth.tar"):
    run("cd hifigan && wget -q https://github.com/ming024/FastSpeech2/raw/master/hifigan/generator_universal.pth.tar.zip"
        " && unzip -o -q generator_universal.pth.tar.zip && rm generator_universal.pth.tar.zip")

Preprocessing: parquet -> wav, train/val selection, MFA corpus and lexicon.

In [ ]:
run(f"python {LANGUAGE}/preprocess.py extract", "preprocess.log")
run(f"python {LANGUAGE}/preprocess.py select --workers {NPROC}", "preprocess.log")

In [ ]:
corpus, lexicon, tg = f"data/{LANGUAGE}/mfa_corpus", f"data/{LANGUAGE}/lexicon.txt", f"data/{LANGUAGE}/TextGrid"
if LANGUAGE == "English":
    run(f"{MFA} model download acoustic english_us_arpa", "mfa.log")
    run(f"{MFA} align {corpus} {lexicon} english_us_arpa {tg} --clean -j {NPROC} --beam 100 --retry_beam 400", "mfa.log")
else:
    # no pretrained Telugu acoustic model, so train one on the selected data (also writes the alignments)
    run(f"{MFA} train {corpus} {lexicon} data/Telugu/telugu_mfa.zip --output_directory {tg} --clean -j {NPROC}", "mfa.log")
print(len(glob.glob(f"{tg}/**/*.TextGrid", recursive=True)), "TextGrids")

In [ ]:
run(f"python {LANGUAGE}/preprocess.py features --workers {NPROC}", "preprocess.log")

Training. Loss curves are rewritten to `Results/<Language>/` at every validation step.

In [ ]:
run(f"python {LANGUAGE}/train.py --steps {TRAIN_STEPS} --max_hours {TRAIN_HOURS}", "train.log")

In [ ]:
run(f"python {LANGUAGE}/inference.py", "inference.log")

from IPython.display import Audio, Image, display
for wav in sorted(glob.glob(f"Results/{LANGUAGE}/test_generated_audios/*.wav")):
    print(os.path.basename(wav))
    display(Audio(wav))
display(Image(f"Results/{LANGUAGE}/training_loss.png"), Image(f"Results/{LANGUAGE}/validation_loss.png"))

Pack everything that goes into the repository.

In [ ]:
info = f"{LANGUAGE}/data_info"
os.makedirs(info, exist_ok=True)
pre = f"data/{LANGUAGE}/preprocessed"
for name in ["selection.tsv", "stats.json", "speakers.json", "train.txt", "val.txt"]:
    shutil.copy(f"{pre}/{name}", info)
shutil.copy(lexicon, info)

zip_path = f"{WORK}/outputs_{LANGUAGE}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for top in [f"{LANGUAGE}/checkpoints", LOGS, info, f"Results/{LANGUAGE}"]:
        for path in glob.glob(f"{top}/**", recursive=True):
            if os.path.isfile(path) and not path.endswith("last.pt"):
                z.write(path)
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")

if ON_COLAB:
    from google.colab import files
    files.download(zip_path)